# ☁️ Project 4 — Remove Clouds from a Sentinel-2 Image

## 🎯 Goal

In this project, you will learn how to:

- Display the original satellite image.
- Remove cloudy pixels using a cloud mask.
- Compare the original and cloud-masked images.
- Understand how image masking works in Google Earth Engine.

---

## 📚 Skills You'll Learn

- ✅ `select()`
- ✅ `eq()`
- ✅ `updateMask()`
- ✅ QA60 Quality Band
- ✅ Image Processing
- ✅ Cloud Masking

---

# 🌍 Why Is Cloud Masking Important?

Imagine you're calculating **NDVI** for a forest.

```
🌳 🌳 🌳 ☁️ ☁️ ☁️ 🌳 🌳
```

The satellite cannot "see" the trees underneath the clouds.

If cloudy pixels remain in your analysis:

- NDVI becomes inaccurate.
- Water detection becomes less reliable.
- Land-cover classification may produce incorrect results.
- Change detection becomes less trustworthy.

For this reason, **cloud masking is one of the first preprocessing steps in most remote sensing workflows.**

---

# 💡 A New Concept: Processing an Image

In the previous projects, you learned how to **select** images.

Now you'll learn how to **modify** an image.

```
Original Image
        ↓
Apply Processing
        ↓
New Image
```

One important rule in Earth Engine:

> **Images are immutable.**

This means processing an image **does not change the original image**.

Instead, Earth Engine creates a **new processed image**.

---

# Step 1 — Create a Study Area

We'll continue using **Berlin**.

```javascript
var cityPoint = ee.Geometry.Point([13.4050, 52.5200]);
```

Nothing new here—you already know how to create a point geometry.

---

# Step 2 — Load a Sentinel-2 Image

```javascript
var image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
  .filterBounds(cityPoint)
  .filterDate('2024-06-01', '2024-06-30')
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first();
```

## Why do we still sort by cloud cover?

Even though we'll remove clouds later, it's still helpful to begin with one of the least cloudy images.

Fewer clouds generally produce a better final result.

---

# Step 3 — Display the Original Image

```javascript
Map.centerObject(cityPoint, 10);

Map.addLayer(
  image,
  {
    bands: ['B4', 'B3', 'B2'],
    min: 0,
    max: 3000
  },
  "Original"
);
```

Nothing new here.

We're simply displaying the original image before applying any processing.

---

# Step 4 — Understanding the QA60 Band

Sentinel-2 contains many bands.

| Band | Purpose |
|------|---------|
| B2 | Blue |
| B3 | Green |
| B4 | Red |
| B8 | Near Infrared |
| QA60 | Quality Information |

Unlike the other bands, **QA60 is not an image you can view as colors.**

Instead, it stores **quality information** about every pixel.

Think of it like an inspection label attached to each pixel.

| Pixel | Quality |
|--------|---------|
| 🌳 Tree | Good |
| 🌊 Water | Good |
| ☁️ Cloud | Cloud |
| ❄️ Snow | Snow |

Internally, QA60 stores this information using **binary flags (bits)**.

For this beginner project, we'll simply check whether the QA60 value equals zero.

---

# Step 5 — Select the QA60 Band

```javascript
var qa = image.select('QA60');
```

## What does `select()` do?

It extracts one or more bands from an image.

Before:

```
Image

B2
B3
B4
...
B8
QA60
```

After:

```
QA60
```

Only the selected band remains.

---

# Step 6 — Build the Cloud Mask

```javascript
var cloudMask = qa.eq(0);
```

This introduces another new function.

## `eq()`

`eq` means **equals**.

This line asks:

> "Is this pixel's QA60 value equal to 0?"

If yes:

```
True
```

If not:

```
False
```

The result is another image containing only **True** and **False** values.

Example:

```
True
False
True
True
False
```

Think of it like a stencil.

```
✔ Keep

✖ Hide

✔ Keep

✔ Keep

✖ Hide
```

---

# Step 7 — Apply the Mask

```javascript
var maskedImage = image.updateMask(cloudMask);
```

## What does `updateMask()` do?

It uses the True/False mask to determine which pixels remain visible.

```
Original Image

🌳 🌳 ☁️ ☁️ 🌳 🌳
```

Mask

```
✔ ✔ ✖ ✖ ✔ ✔
```

Result

```
🌳 🌳     🌳 🌳
```

Cloudy pixels become transparent.

---

# Step 8 — Display the Result

```javascript
Map.addLayer(
  maskedImage,
  {
    bands: ['B4', 'B3', 'B2'],
    min: 0,
    max: 3000
  },
  "Cloud Masked"
);
```

Now you can compare:

- Original image
- Cloud-masked image

Turn the layers on and off in the **Layers** panel to see the difference.

---

# Complete Code

```javascript
// Study area
var cityPoint = ee.Geometry.Point([13.4050, 52.5200]);

// Get one image
var image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
  .filterBounds(cityPoint)
  .filterDate('2024-06-01', '2024-06-30')
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first();

// Center map
Map.centerObject(cityPoint, 10);

// Visualization settings
var trueColor = {
  bands: ['B4', 'B3', 'B2'],
  min: 0,
  max: 3000
};

// Display original image
Map.addLayer(image, trueColor, "Original");

// Select QA60 band
var qa = image.select('QA60');

// Build cloud mask
var cloudMask = qa.eq(0);

// Apply mask
var maskedImage = image.updateMask(cloudMask);

// Display masked image
Map.addLayer(maskedImage, trueColor, "Cloud Masked");
```

---

# Workflow Summary

```
Sentinel-2 Image
        ↓
Select QA60 Band
        ↓
Check Which Pixels Equal 0
        ↓
Create True/False Mask
        ↓
Apply updateMask()
        ↓
Cloud-Masked Image
```

---

# Key Concepts

## ✅ `select()`

Extracts one or more bands from an image.

Example:

```javascript
image.select('B4')
```

---

## ✅ `eq()`

Compares pixel values.

Example:

```javascript
qa.eq(0)
```

returns:

```
True
False
```

for every pixel.

---

## ✅ `updateMask()`

Uses a True/False image to determine which pixels remain visible.

```
True  → Keep

False → Hide
```

---

## ✅ Images Are Immutable

This code:

```javascript
var maskedImage = image.updateMask(cloudMask);
```

creates a **new image**.

The original image is still available and unchanged.

---

# Real-World Note

The simple mask:

```javascript
qa.eq(0)
```

is perfect for learning, but it is **not** the method most researchers use.

In professional workflows, cloud masking is usually performed by:

- Checking the specific cloud and cirrus bits in the QA60 band.
- Using the **Sentinel-2 Cloud Probability** dataset.
- Using **Cloud Score+**, which provides more accurate cloud detection.

We're starting with the simple approach because the goal of this lesson is to understand:

- What a mask is.
- How `select()`, `eq()`, and `updateMask()` work.
- How image processing creates a new image.

Later in this course, you'll learn the professional cloud-masking methods used in research.

---

# Practice Exercise

Choose another city (or use Isfahan).

Your project should:

- ✅ Create a point geometry.
- ✅ Load a Sentinel-2 image.
- ✅ Display the original image.
- ✅ Select the QA60 band.
- ✅ Create a cloud mask using `eq(0)`.
- ✅ Apply the mask using `updateMask()`.
- ✅ Display the cloud-masked image.

---

# 🚀 Bonus Challenge

Create a **False Color** version of the cloud-masked image using:

```javascript
bands: ['B8', 'B4', 'B3']
```

Compare the original and cloud-masked images to see how vegetation becomes easier to observe after cloudy pixels are removed.